# 01 · Panel SKU × día completo

`train.csv` solo trae filas para los días en que un SKU vendió. Este notebook construye el **panel completo**: para cada SKU, todos los días con operación de la tienda entre su primera y su última venta (su *ventana de vida*), con venta cero en los días sin registro.

| | |
|---|---|
| **Entrada** | `../data/procesados/t44_registros.parquet` (de `00_extraccion_t44`) |
| **Salida** | `../data/procesados/panel_t44_sku_dia.parquet` |
| **Decisiones que implementa** | D‑03 (unidad SKU × día), D‑20 (ventas negativas), D‑24 (universo de SKUs), D‑25 (promoción en días sin venta) |

**Por qué importa.** Las medias y comparaciones calculadas solo sobre los días con venta están condicionadas a que hubo venta (M‑03): excluyen justamente los días en que la promoción podría haber hecho la diferencia entre vender y no vender.

In [4]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook construye el panel completo SKU x día de la tienda 44, con ceros en los días sin venta.
# ------------------------------------------------------------------------------------------------------------
# Implementación: Ramiro Saltos
# Fecha de versión: 2026-10-15
# Revisado: 2026-10-15
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os

import numpy as np
import pandas as pd

In [5]:
# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_procesados = os.path.join("..", "data", "procesados")
archivo_entrada = os.path.join(ruta_procesados, "t44_registros.parquet")
archivo_salida = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")

# A-01: largo máximo (días de operación) de una racha sin venta para imputar promoción
max_racha_imputacion = 7

# Valor esperado para el diseño acordado (verificado el 2026-10-15)
celdas_esperadas = 1_935_758

registros = pd.read_parquet(archivo_entrada)
print(f"Registros leídos: {len(registros):,}")

Registros leídos: 1,599,702


## 1. Días con operación y ventana de vida de cada SKU (D‑24)

Los días con operación son los días en que la tienda registró alguna venta (590; no abre el 25 de diciembre ni el 1 de enero). La ventana de vida de un SKU va de su primera a su última venta en el período. **No se filtran SKUs por frecuencia de venta**: el filtro Pareto del EDA anterior seleccionaba sobre el resultado (M‑04, R‑05).

**Supuestos que deben declararse en la tesis:**

- **Ventas = demanda.** Los días sin venta dentro de la ventana de vida se registran con venta cero, también en las rachas largas: muchos SKUs son de temporada y luego salen de la oferta (A‑02). No hay información de quiebres de stock.
- **La ventana de vida usa información del futuro**: su fin es la última venta del SKU, que solo se conoce al terminar el período (A‑10).

In [6]:
# Obtener los días con operación de la tienda
dias_operacion = pd.DataFrame({"date": np.sort(registros["date"].unique())})

# Calcular la ventana de vida y los atributos fijos de cada SKU
skus = (
    registros.groupby("item_nbr", as_index=False)
    .agg(
        fecha_primera=("date", "min"),
        fecha_ultima=("date", "max"),
        family=("family", "first"),
        clase=("class", "first"),
        perishable=("perishable", "first"),
    )
    .rename(columns={"clase": "class"})
)

print(f"Días con operación: {len(dias_operacion)}")
print(f"SKUs:               {len(skus):,}")

Días con operación: 590
SKUs:               3,656


## 2. Panel completo

Se cruzan todos los SKUs con todos los días con operación y se conservan las combinaciones dentro de la ventana de vida de cada SKU. Luego se agregan las ventas y la promoción observadas; donde no hay registro, la venta es cero y la promoción queda sin observar.

In [7]:
def construir_panel_completo(registros: pd.DataFrame, skus: pd.DataFrame, dias: pd.DataFrame) -> pd.DataFrame:
    """
    Esta función construye el panel SKU x día dentro de la ventana de vida de cada SKU, con venta cero en los días
    sin registro.

    :param registros:   un DataFrame con una fila por SKU-día con registro (date, item_nbr, unit_sales, onpromotion).
    :param skus:        un DataFrame con una fila por SKU, su ventana de vida (fecha_primera, fecha_ultima) y sus
                        atributos (family, class, perishable).
    :param dias:        un DataFrame con la columna date de los días con operación de la tienda.
    :return:            un DataFrame con una fila por SKU-día dentro de la ventana de vida, ordenado por SKU y fecha.
    """

    # Cruzar SKUs y días, y conservar solo los días dentro de la ventana de vida de cada SKU
    panel = skus.merge(dias, how="cross")
    en_ventana = (panel["date"] >= panel["fecha_primera"]) & (panel["date"] <= panel["fecha_ultima"])
    panel = panel.loc[en_ventana].drop(columns=["fecha_primera", "fecha_ultima"])

    # Agregar las ventas y la promoción observadas
    observados = registros[["item_nbr", "date", "unit_sales", "onpromotion"]].rename(
        columns={"onpromotion": "onpromotion_obs"}
    )
    panel = panel.merge(observados, on=["item_nbr", "date"], how="left")

    # Marcar los días con registro y completar la venta con cero en los días sin registro
    panel["con_registro"] = panel["unit_sales"].notna()
    panel["unit_sales"] = panel["unit_sales"].fillna(0).astype("float32")

    return panel.sort_values(["item_nbr", "date"]).reset_index(drop=True)


panel = construir_panel_completo(registros, skus, dias_operacion)

# Verificar la granularidad y el tamaño esperado
assert not panel.duplicated(["item_nbr", "date"]).any(), "hay más de una fila por SKU-día"
assert panel["con_registro"].sum() == len(registros), "se perdieron registros al construir el panel"
assert len(panel) == celdas_esperadas, f"se esperaban {celdas_esperadas:,} celdas y hay {len(panel):,}"

print(f"Celdas SKU-día:        {len(panel):,}")
print(f"Con registro:          {panel['con_registro'].sum():,}")
print(f"Sin venta (completas): {(~panel['con_registro']).sum():,}  ({(~panel['con_registro']).mean():.1%})")

Celdas SKU-día:        1,935,758
Con registro:          1,599,702
Sin venta (completas): 336,056  (17.4%)


## 3. Ventas negativas (D‑20)

Las ventas negativas son devoluciones. Se conservan en `unit_sales` (ventas netas, para agregados) y se crea `unit_sales_ppml`, truncada en cero, porque PPML exige una variable dependiente no negativa.

In [8]:
# Crear la venta truncada en cero para PPML
panel["unit_sales_ppml"] = panel["unit_sales"].clip(lower=0).astype("float32")

n_negativas = (panel["unit_sales"] < 0).sum()
print(f"Celdas con venta negativa truncadas a cero: {n_negativas:,}")

Celdas con venta negativa truncadas a cero: 210


## 4. Promoción en los días sin venta (D‑25)

En los días sin registro, `onpromotion` no se observa. Como las campañas duran varios días, se imputa así:

- si el día observado **anterior** y el **siguiente** del mismo SKU tienen el mismo estado **y la racha sin venta dura como máximo 7 días de operación**, se usa ese estado;
- si difieren (casos mixtos) o la racha es más larga, se supone **sin promoción**: una campaña de varias semanas sin ninguna venta no es plausible (A‑01).

Se guardan además dos cotas para el análisis de robustez: la **inferior** supone sin promoción en todos los días sin observar, y la **superior** supone promoción cuando el día anterior o el siguiente la tiene. La columna `promo_imputada` marca los días cuyo estado no fue observado.

In [9]:
def imputar_promocion(panel: pd.DataFrame, max_racha: int) -> pd.DataFrame:
    """
    Esta función imputa la promoción en los días sin registro (D-25) y agrega las cotas inferior y superior.

    :param panel:       un DataFrame ordenado por SKU y fecha, con la columna onpromotion_obs (nula en días sin
                        registro).
    :param max_racha:   el largo máximo, en días de operación, de una racha sin venta en la que se imputa promoción.
    :return:            el mismo DataFrame con las columnas onpromotion (imputación principal), onpromotion_cota_inf,
                    onpromotion_cota_sup y promo_imputada.
    """

    # Pasar la promoción observada a 1/0 con nulos en los días sin registro
    observada = panel["onpromotion_obs"].astype("float")
    faltante = observada.isna()

    # Obtener el estado observado anterior y siguiente dentro de cada SKU
    anterior = observada.groupby(panel["item_nbr"]).ffill()
    siguiente = observada.groupby(panel["item_nbr"]).bfill()

    # La ventana de vida empieza y termina con un registro, así que siempre hay estado anterior y siguiente
    assert anterior[faltante].notna().all() and siguiente[faltante].notna().all(), "falta un vecino observado"

    # Medir el largo de cada racha de días sin registro dentro de cada SKU
    inicio_racha = faltante & ~faltante.groupby(panel["item_nbr"]).shift(fill_value=False)
    id_racha = inicio_racha.cumsum().where(faltante)
    largo_racha = id_racha.map(id_racha.value_counts())

    # Imputar: el estado común si los vecinos coinciden y la racha es corta; si no, sin promoción
    coinciden = (anterior == siguiente) & (largo_racha <= max_racha)
    principal = observada.where(~faltante, np.where(coinciden, anterior, 0.0))
    cota_superior = observada.where(~faltante, ((anterior == 1) | (siguiente == 1)).astype(float))

    panel["onpromotion"] = principal.astype(bool)
    panel["onpromotion_cota_inf"] = observada.fillna(0).astype(bool)
    panel["onpromotion_cota_sup"] = cota_superior.astype(bool)
    panel["promo_imputada"] = faltante

    return panel


panel = imputar_promocion(panel, max_racha_imputacion)

# Resumir los casos de la imputación
sin_registro = ~panel["con_registro"]
imputadas_con = panel["onpromotion"] & sin_registro
no_imputadas = sin_registro & panel["onpromotion_cota_sup"] & ~panel["onpromotion"]
resto_sin = sin_registro & ~panel["onpromotion_cota_sup"]
total = sin_registro.sum()
casos = {
    "imputados con promoción":                           imputadas_con.sum(),
    "vecino con promoción, pero mixto o racha larga":    no_imputadas.sum(),
    "entre dos días sin promoción":                      resto_sin.sum(),
}
print(f"Días sin registro: {total:,}")
for nombre, n in casos.items():
    print(f"  {nombre:<50} {n:>9,}  ({n / total:.1%})")

Días sin registro: 336,056
  imputados con promoción                                7,221  (2.1%)
  vecino con promoción, pero mixto o racha larga        25,466  (7.6%)
  entre dos días sin promoción                         303,369  (90.3%)


## 5. Resumen por familia y exportación

In [10]:
# Resumir el panel por familia
resumen = (
    panel.groupby("family", observed=True)
    .agg(
        skus=("item_nbr", "nunique"),
        celdas=("item_nbr", "size"),
        pct_sin_venta=("con_registro", lambda s: 100 * (1 - s.mean())),
        pct_en_promocion=("onpromotion", lambda s: 100 * s.mean()),
    )
    .sort_values("celdas", ascending=False)
    .round(1)
)
print(resumen.to_string())

# Ordenar las columnas y guardar
columnas = [
    "date", "item_nbr", "family", "class", "perishable",
    "unit_sales", "unit_sales_ppml", "con_registro",
    "onpromotion_obs", "onpromotion", "onpromotion_cota_inf", "onpromotion_cota_sup", "promo_imputada",
]
panel = panel[columnas]
panel.to_parquet(archivo_salida, index=False)
print(f"\nGuardado: {archivo_salida}  ({os.path.getsize(archivo_salida) / 1024**2:.1f} MB)")

print(f"\n>>>> Proceso finalizado <<<<")

                            skus  celdas  pct_sin_venta  pct_en_promocion
family                                                                   
GROCERY I                   1176  620919           15.1               6.8
BEVERAGES                    581  287407           15.2               7.3
CLEANING                     399  220098           18.0               6.5
PRODUCE                      265  148802           12.1              14.7
DAIRY                        229  122674            5.1               9.3
PERSONAL CARE                148   79578           28.0               6.9
BREAD/BAKERY                 108   60669            5.5               5.8
HOME CARE                     98   51281           13.0               8.0
DELI                          77   43958            6.7              21.1
HOME AND KITCHEN I            76   39710           65.2               6.0
MEATS                         58   32009            4.7              17.7
POULTRY                       50   276